#Transulate articles

In [ ]:
!pip install googletrans==4.0.0-rc1

#Combine csv annd run transulation

In [ ]:
import pandas as pd
import os
from googletrans import Translator

# -----------------------------
# Configuration
# -----------------------------
folder_path = "/content/drive/MyDrive/Venice_RC15/Datasets/News"
output_file = "/content/drive/MyDrive/Venice_RC15/Datasets/News/combined_news_cleaned.csv"

translator = Translator()

# -----------------------------
# Combine CSV files
# -----------------------------
all_dfs = []

for file_name in os.listdir(folder_path):
    if file_name.endswith(".csv"):
        file_path = os.path.join(folder_path, file_name)
        df = pd.read_csv(file_path)

        print(f"{file_name}: {len(df)} datapoints")

        # Add CSV filename column at the start
        df.insert(0, "source_file", file_name)

        all_dfs.append(df)

# Concatenate all data
combined_df = pd.concat(all_dfs, ignore_index=True)

# -----------------------------
# Fill empty cells with 'NA'
# -----------------------------
combined_df.fillna("NA", inplace=True)

# -----------------------------
# Translate Italian to English for non-English text
# We'll assume 'title' and 'snippet' columns may need translation
# -----------------------------
for col in ['title', 'snippet', 'desc']:
    if col in combined_df.columns:
        def translate_text(text):
            if text == "NA":
                return text
            try:
                translation = translator.translate(text, src='auto', dest='en')
                return translation.text
            except:
                return text  # fallback if translation fails

        combined_df[col] = combined_df[col].apply(translate_text)

# -----------------------------
# Remove duplicates based on 'link' or 'title'
# -----------------------------
if 'link' in combined_df.columns:
    combined_df.drop_duplicates(subset='link', inplace=True)
elif 'title' in combined_df.columns:
    combined_df.drop_duplicates(subset='title', inplace=True)

# -----------------------------
# Export cleaned CSV
# -----------------------------
combined_df.to_csv(output_file, index=False)
print(f"Cleaned combined CSV saved at: {output_file}")
print(f"Total datapoints after cleaning: {len(combined_df)}")
